# Data Description
- Author: 待學生填寫（已詢問）
- Date first created: 原始資料建立日未提供；本說明建立於 2026-11-08
- Background: 使用者指定的 HI／VT 房貸申請整理檔；檔名含 white_percentage，但實際欄位涵蓋多種族申請紀錄，沒有白人人口比例欄位。
- Unit of observation: 一列為一筆提供的申請紀錄；缺少申請識別碼，無法驗證唯一申請或同一人重複申請。
- Sample period: 未提供；收入門檻為 2024 ACS 不代表房貸樣本年份。
- Known issues: 原始下載來源、建立日期與上游程式未提供。income 缺失者不納入收入組分析；不依整列相同而去重。收入單位及申請結果編碼沿用既有 MA／MS 分析，尚未以原始下載紀錄獨立驗證。
- Definition for each variable:
    - state_code／county_code：州別／郡代碼；derived_ethnicity／derived_race／derived_sex：檔案提供的彙整族裔、種族與性別分類。
    - action_taken：申請結果，沿用 1 為已撥款、2 為核准未接受、3 為拒絕；本次核准為 1 或 2。
    - loan_type／loan_purpose／occupancy_type：貸款類型、用途與居住用途分類代碼。
    - loan_amount／property_value：貸款金額／房產價值（沿用美元）；loan_to_value_ratio：貸款價值比（沿用百分比）。
    - income：年收入，沿用千美元；debt_to_income_ratio：負債所得比，包含分組或其他文字代碼；applicant_age：申請人年齡組。
- Data path: `data/processed/state_HI-VT_white_percentage_15_columns.csv`
- Source data: 使用者提供的外部整理檔，原始下載來源未提供。
- Processing code: not available（未提供上游程式）
- Role: analysis
- Verification status: 2026-11-08 已由乾淨 Jupyter kernel 完整執行；39,292 列、15 欄、HI／VT 州別、申請結果 1／2／3、必要欄位與 hash 均通過；income 缺失 2,323 筆；年份、唯一申請、來源與單位的獨立驗證仍未確認。
- SHA-256: `962fb1339dbc92effc07c0d793089307d05f22fc78e01d6daf63741ffd7d9da6`
- Downstream analysis input: `data/processed/state_HI-VT_white_percentage_15_columns.csv`；[HI／VT 分析](../../finding/hi_vt_mortgage_approval.ipynb)


In [1]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual


## Change for Your Own Data
以下精確綁定使用者指定檔案，唯讀驗證，不更動資料。


In [2]:
INPUT_RELATIVE_PATH = "data/processed/state_HI-VT_white_percentage_15_columns.csv"
input_data_file = project_file(INPUT_RELATIVE_PATH, "processed")
EXPECTED_COLUMNS = ['state_code', 'county_code', 'derived_ethnicity', 'derived_race', 'derived_sex', 'action_taken', 'loan_type', 'loan_purpose', 'loan_amount', 'loan_to_value_ratio', 'property_value', 'occupancy_type', 'income', 'debt_to_income_ratio', 'applicant_age']
EXPECTED_ROWS = 39292
EXPECTED_SHA256 = "962fb1339dbc92effc07c0d793089307d05f22fc78e01d6daf63741ffd7d9da6"


## Summary


In [3]:
input_fingerprint = verify_input_sha256(input_data_file, EXPECTED_SHA256)
df = pd.read_csv(input_data_file)  # Configure dtype/parse_dates/na_values for the actual file.
print({"data_path": INPUT_RELATIVE_PATH, "rows": len(df),
       "schema": {column: str(dtype) for column, dtype in df.dtypes.items()},
       "sha256": sha256_file(input_data_file)})


{'data_path': 'data/processed/state_HI-VT_white_percentage_15_columns.csv', 'rows': 39292, 'schema': {'state_code': 'str', 'county_code': 'float64', 'derived_ethnicity': 'str', 'derived_race': 'str', 'derived_sex': 'str', 'action_taken': 'int64', 'loan_type': 'int64', 'loan_purpose': 'int64', 'loan_amount': 'float64', 'loan_to_value_ratio': 'str', 'property_value': 'str', 'occupancy_type': 'int64', 'income': 'float64', 'debt_to_income_ratio': 'str', 'applicant_age': 'str'}, 'sha256': '962fb1339dbc92effc07c0d793089307d05f22fc78e01d6daf63741ffd7d9da6'}


### Sample


In [4]:
display(df.head(10))

,state_code,county_code,derived_ethnicity,derived_race,derived_sex,action_taken,loan_type,loan_purpose,loan_amount,loan_to_value_ratio,property_value,occupancy_type,income,debt_to_income_ratio,applicant_age
0,HI,15009.0,Ethnicity Not Available,Race Not Available,Joint,3,1,1,795000.0,80.00000,995000,1,175.0,38,35-44
1,HI,15003.0,Not Hispanic or Latino,White,Female,1,3,1,515000.0,99.99800,515000,1,250.0,<20%,25-34
2,HI,15003.0,Hispanic or Latino,Asian,Male,1,3,1,555000.0,90.00400,625000,1,113.0,30%-<36%,25-34
3,HI,15001.0,Joint,Joint,Joint,1,3,1,725000.0,99.31700,735000,1,190.0,36,45-54
4,HI,15003.0,Hispanic or Latino,Race Not Available,Male,1,3,1,545000.0,100.00000,545000,1,145.0,45,25-34
5,VT,50011.0,Not Hispanic or Latino,White,Male,1,3,32,325000.0,90.00000,365000,1,86.0,49,55-64
6,VT,50021.0,Hispanic or Latino,Race Not Available,Male,1,3,31,255000.0,NaN,NaN,1,NaN,NaN,35-44
7,VT,50017.0,Not Hispanic or Latino,White,Joint,1,2,32,285000.0,80.00000,345000,1,158.0,42,45-54
8,HI,15003.0,Not Hispanic or Latino,Asian,Male,1,3,1,815000.0,100.00000,815000,1,161.0,>60%,35-44
9,HI,15003.0,Ethnicity Not Available,Race Not Available,Male,1,3,1,865000.0,97.19100,895000,1,180.0,46,35-44


### Summary Stats


In [5]:
import numpy as np
from IPython.display import display
print(f'表 1：全欄位摘要（總 N={len(df):,}）')
display(df.describe(include='all'))
print('表 2：各欄缺失筆數')
display(df.isna().sum().to_frame('缺失筆數'))
for col in ['state_code','derived_race','action_taken']:
    counts=df[col].value_counts()
    print(f'表：{col}（有效 N={counts.sum():,}）')
    display(pd.DataFrame({'筆數':counts,'比例 (%)':counts/counts.sum()*100}))
income=pd.to_numeric(df.income,errors='coerce')
print('收入非有限／缺失筆數：',int((~np.isfinite(income)).sum()))


表 1：全欄位摘要（總 N=39,292）


,state_code,county_code,derived_ethnicity,derived_race,derived_sex,action_taken,loan_type,loan_purpose,loan_amount,loan_to_value_ratio,property_value,occupancy_type,income,debt_to_income_ratio,applicant_age
count,39292,39166.000000,39292,39292,39292,39292.000000,39292.000000,39292.000000,3.929200e+04,37127,37978,39292.000000,36969.000000,35935,39292
unique,2,NaN,5,9,4,NaN,NaN,NaN,NaN,19587,548,NaN,NaN,20,8
top,HI,NaN,Not Hispanic or Latino,White,Joint,NaN,NaN,NaN,NaN,80,355000,NaN,NaN,30%-<36%,35-44
freq,24196,NaN,28687,17682,15959,NaN,NaN,NaN,NaN,1687,578,NaN,NaN,4231,9205
mean,NaN,28432.734157,NaN,NaN,NaN,1.548661,1.258806,9.810038,4.086789e+05,NaN,NaN,1.296956,197.481944,NaN,NaN
std,NaN,17024.419583,NaN,NaN,NaN,0.874328,0.638160,13.224948,7.133118e+05,NaN,NaN,0.670975,466.085362,NaN,NaN
min,NaN,15001.000000,NaN,NaN,NaN,1.000000,1.000000,1.000000,5.000000e+03,NaN,NaN,1.000000,-4695.000000,NaN,NaN
25%,NaN,15003.000000,NaN,NaN,NaN,1.000000,1.000000,1.000000,1.250000e+05,NaN,NaN,1.000000,84.000000,NaN,NaN
50%,NaN,15003.000000,NaN,NaN,NaN,1.000000,1.000000,2.000000,2.850000e+05,NaN,NaN,1.000000,130.000000,NaN,NaN
75%,NaN,50007.000000,NaN,NaN,NaN,3.000000,1.000000,31.000000,5.150000e+05,NaN,NaN,1.000000,200.000000,NaN,NaN


表 2：各欄缺失筆數


,缺失筆數
state_code,0
county_code,126
derived_ethnicity,0
derived_race,0
derived_sex,0
action_taken,0
loan_type,0
loan_purpose,0
loan_amount,0
loan_to_value_ratio,2165


表：state_code（有效 N=39,292）


,筆數,比例 (%)
state_code,,
HI,24196,61.579965
VT,15096,38.420035


表：derived_race（有效 N=39,292）


,筆數,比例 (%)
derived_race,,
White,17682,45.001527
Asian,8085,20.576708
Race Not Available,7407,18.851166
Joint,2441,6.212461
Native Hawaiian or Other Pacific Islander,1803,4.588720
2 or more minority races,1161,2.954800
Black or African American,517,1.315789
American Indian or Alaska Native,194,0.493739
Free Form Text Only,2,0.005090


表：action_taken（有效 N=39,292）


,筆數,比例 (%)
action_taken,,
1,27887,70.973735
3,10153,25.839866
2,1252,3.186399


收入非有限／缺失筆數： 2323


### Assertions


In [6]:
# Replace the schema placeholder; add checks for the actual keys, ranges,
# missingness, units, merge coverage, and train/test separation as applicable.
if not EXPECTED_COLUMNS or any("<" in name for name in EXPECTED_COLUMNS):
    raise ValueError("Fill the expected schema and dataset-specific assertions before verification.")
assert list(df.columns) == EXPECTED_COLUMNS, "Processed schema differs from the documented schema."
if EXPECTED_ROWS is not None:
    assert len(df) == EXPECTED_ROWS, "Processed row count differs from its processing record."
actual_sha256 = sha256_file(input_data_file)
assert actual_sha256 == input_fingerprint, "File changed during documentation: inspect the new version."
print({"data_path": INPUT_RELATIVE_PATH, "rows": len(df), "columns": list(df.columns),
       "sha256": actual_sha256, "status": "configured checks passed"})
# Update Data Description and the PROJECT.md Data Artifacts row only from the
# real results. Store the same exact path and verified SHA-256 in downstream
# finding code; do not rename/copy this file merely to match another example.

assert set(df.state_code)=={'HI','VT'}
assert set(df.action_taken)=={1,2,3}
assert df[['state_code','derived_race','action_taken']].notna().all().all()
print('州別、必要欄位、39,292 列／15 欄、結果代碼及檔案 hash 通過；年份、唯一申請與來源尚未驗證。')


{'data_path': 'data/processed/state_HI-VT_white_percentage_15_columns.csv', 'rows': 39292, 'columns': ['state_code', 'county_code', 'derived_ethnicity', 'derived_race', 'derived_sex', 'action_taken', 'loan_type', 'loan_purpose', 'loan_amount', 'loan_to_value_ratio', 'property_value', 'occupancy_type', 'income', 'debt_to_income_ratio', 'applicant_age'], 'sha256': '962fb1339dbc92effc07c0d793089307d05f22fc78e01d6daf63741ffd7d9da6', 'status': 'configured checks passed'}
州別、必要欄位、39,292 列／15 欄、結果代碼及檔案 hash 通過；年份、唯一申請與來源尚未驗證。
